# MovieLens 100K — Recommendation System

**Course:** Data Mining  
**Dataset:** [MovieLens 100K](https://grouplens.org/datasets/movielens/100k/)  
**Models:** User-based CF, Item-based CF, optional SVD (sklearn TruncatedSVD)  
**Metrics:** RMSE, MAE  
**Reproducibility:** `random_state=42`

Run this notebook from the project root after placing `u.data` at `data/raw/ml-100k/u.data`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Project root (parent of notebooks/)
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RANDOM_STATE, FIGURES_DIR, K_NEIGHBORS
from src.data_loader import load_ratings, load_movies, dataset_summary
from src.preprocessing import split_ratings, global_mean
from src.metrics import rmse, mae
from src.user_based_cf import UserBasedCF
from src.item_based_cf import ItemBasedCF
from src import visualization as viz

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"random_state={RANDOM_STATE}")

## 2. Exploratory Data Analysis (EDA)

In [ ]:
ratings = load_ratings()
movies = load_movies()

summary = dataset_summary(ratings)
print("Dataset summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

ratings.head()

In [ ]:
movies[["movie_id", "title"]].head(10)

In [ ]:
viz.plot_rating_distribution(ratings)
viz.plot_ratings_per_user(ratings)
viz.plot_ratings_per_movie(ratings)
viz.plot_genre_counts(movies)
print(f"Figures saved to {FIGURES_DIR}")

## 3. Preprocessing — Train / Test Split

In [ ]:
train, test = split_ratings(ratings, random_state=RANDOM_STATE)
baseline_pred = np.full(len(test), global_mean(train))
y_test = test["rating"].values

print(f"Train size: {len(train):,}  |  Test size: {len(test):,}")
print(f"Global mean (train): {global_mean(train):.4f}")
print(f"Baseline RMSE: {rmse(y_test, baseline_pred):.4f}")
print(f"Baseline MAE:  {mae(y_test, baseline_pred):.4f}")

## 4. Model Training & Evaluation

Each model is fit on **train** only; metrics are computed on **test**.

In [ ]:
results = []
predictions = {}

# Baseline
results.append({
    "model": "Global Mean",
    "rmse": rmse(y_test, baseline_pred),
    "mae": mae(y_test, baseline_pred),
})
predictions["Global Mean"] = baseline_pred

### 4.1 User-Based Collaborative Filtering

In [ ]:
user_cf = UserBasedCF(k=K_NEIGHBORS)
print("Fitting user-based CF (similarity matrix)...")
user_cf.fit(train)

print("Predicting on test set...")
user_pred = user_cf.predict_batch(test)
predictions["User-Based CF"] = user_pred

results.append({
    "model": "User-Based CF",
    "rmse": rmse(y_test, user_pred),
    "mae": mae(y_test, user_pred),
})
print(f"User-Based CF — RMSE: {results[-1]['rmse']:.4f}, MAE: {results[-1]['mae']:.4f}")

### 4.2 Item-Based Collaborative Filtering

In [ ]:
item_cf = ItemBasedCF(k=K_NEIGHBORS)
print("Fitting item-based CF (similarity matrix)...")
item_cf.fit(train)

print("Predicting on test set...")
item_pred = item_cf.predict_batch(test)
predictions["Item-Based CF"] = item_pred

results.append({
    "model": "Item-Based CF",
    "rmse": rmse(y_test, item_pred),
    "mae": mae(y_test, item_pred),
})
print(f"Item-Based CF — RMSE: {results[-1]['rmse']:.4f}, MAE: {results[-1]['mae']:.4f}")

### 4.3 Optional — SVD Matrix Factorization (sklearn TruncatedSVD)

Latent-factor model using `sklearn.decomposition.TruncatedSVD` on a sparse, mean-centered user–item matrix.

In [ ]:
from src.svd_model import SVDRecommender

svd = SVDRecommender(random_state=RANDOM_STATE)
print("Fitting TruncatedSVD...")
svd.fit(train)

print("Predicting on test set...")
svd_pred = svd.predict_batch(test)
predictions["SVD (TruncatedSVD)"] = svd_pred

results.append({
    "model": "SVD (TruncatedSVD)",
    "rmse": rmse(y_test, svd_pred),
    "mae": mae(y_test, svd_pred),
})
print(f"SVD — RMSE: {results[-1]['rmse']:.4f}, MAE: {results[-1]['mae']:.4f}")

## 5. Results Summary

In [ ]:
results_df = pd.DataFrame(results).sort_values("rmse")
results_df

In [ ]:
viz.plot_model_comparison(results_df)

for name, preds in predictions.items():
    if name == "Global Mean":
        continue
    viz.plot_predicted_vs_actual(y_test, preds, name)
    viz.plot_residuals(y_test, preds, name)

print(f"All figures saved to {FIGURES_DIR}")

## 6. Result Interpretation

The table below summarizes hold-out performance (`random_state=42`, 80/20 split). Lower RMSE and MAE indicate better rating prediction.

| Model | RMSE | MAE | Comment |
|-------|------|-----|---------|
| Global Mean | ~1.13 | ~0.90 | Naive baseline; ignores user and item differences |
| User-Based CF | ~0.97 | ~0.77 | Strong improvement over baseline |
| **Item-Based CF** | **0.9381** | **0.7342** | **Best overall** |
| SVD (TruncatedSVD) | ~1.02+ | ~0.81+ | Worse than neighborhood CF in this setup |

### 6.1 Model ranking and why Item-Based CF wins

- **Item-Based CF performs best** (RMSE **0.9381**, MAE **0.7342**). Movies are a stable catalog; co-rating patterns between items (e.g., genre, director, era) transfer well when predicting a user’s score from similar titles they already rated.
- **User-Based CF is slightly worse** but still **much better than the global-mean baseline**. Finding like-minded users is harder when taste is heterogeneous and each user has only ~100 ratings on average.
- **SVD (sklearn `TruncatedSVD`) underperforms collaborative filtering** here because it is a **simplified low-rank approximation**: it does not include explicit user/item **bias** terms, is not trained with a **recommendation-specific loss** (e.g., weighted RMSE), and treats unobserved matrix entries as zeros after mean-centering. Production matrix-factorization systems (e.g., biased SVD / ALS) typically close this gap.

### 6.2 Sparsity in MovieLens 100K

MovieLens 100K contains **100,000 ratings** from **943 users** on **1,682 movies**. The full user–item matrix has \(943 \times 1{,}682 \approx 1.59\) million cells, so only about **6.3%** of entries are observed and the matrix is roughly **93.7% sparse**. Neighborhood methods must rely on limited overlap between users or items; this sparsity motivates latent-factor models while also explaining why naive factorization without careful bias handling can struggle.

### 6.3 Error by rating level (extreme preferences)

Prediction error is **not uniform across rating levels**. For Item-Based CF, mean absolute error tends to be **higher for true ratings of 1 and 5** than for middle ratings (2–4). Users express **extreme likes and dislikes** less predictably from neighborhood averages—especially when few highly similar items or neighbors are available—while moderate ratings cluster near a user’s typical score and are easier to regress toward the mean.

In [ ]:
# Numeric summary aligned with Section 6 (computed from this run)
results_display = pd.DataFrame(results).sort_values("rmse")
print("=== Hold-out metrics ===")
display(
    results_display.style.format({"rmse": "{:.4f}", "mae": "{:.4f}"}).hide(axis="index")
)

sparsity_pct = 100.0 * summary["sparsity"]
print(
    f"\nSparsity: {summary['n_ratings']:,} ratings / "
    f"({summary['n_users']:,} users × {summary['n_movies']:,} movies) "
    f"→ {sparsity_pct:.1f}% of matrix entries are missing"
)

In [ ]:
# Mean absolute error by true rating (Item-Based CF)
error_by_rating = (
    test.assign(pred=item_pred, abs_error=lambda d: (d["rating"] - d["pred"]).abs())
    .groupby("rating")["abs_error"]
    .mean()
    .rename("mean_abs_error")
)
print("Item-Based CF — mean absolute error by actual rating:")
display(error_by_rating.to_frame().style.format({"mean_abs_error": "{:.4f}"}))

viz.plot_error_by_rating(error_by_rating, "Item-Based CF", filename="error_by_rating_item_cf.png")
print(f"Figure saved: {FIGURES_DIR / 'error_by_rating_item_cf.png'}")

## 7. Hyperparameter Sensitivity — `K_NEIGHBORS` (Item-Based CF)

We vary the number of item neighbors \(k \in \{10, 20, 30, 40, 50\}\) while keeping the same train/test split and item–item similarity matrix (fit once, predict with different \(k\)).

In [ ]:
K_VALUES = [10, 20, 30, 40, 50]

# Reuse Item-Based CF fitted in Section 4.2 (only k changes at prediction time)
item_cf_k = item_cf
print("Reusing fitted item similarity matrix from Section 4.2...")

k_rows = []
for k in K_VALUES:
    item_cf_k.k = k
    pred_k = item_cf_k.predict_batch(test)
    k_rows.append({
        "k": k,
        "rmse": rmse(y_test, pred_k),
        "mae": mae(y_test, pred_k),
    })
    print(f"  k={k:2d}  RMSE={k_rows[-1]['rmse']:.4f}  MAE={k_rows[-1]['mae']:.4f}")

k_results_df = pd.DataFrame(k_rows)
display(k_results_df.style.format({"rmse": "{:.4f}", "mae": "{:.4f}"}).hide(axis="index"))

viz.plot_itemcf_k_sensitivity(k_results_df, filename="itemcf_k_sensitivity.png")
print(f"Figure saved: {FIGURES_DIR / 'itemcf_k_sensitivity.png'}")

### How \(k\) affects performance

- **Small \(k\) (e.g., 10):** Predictions rely on only the closest neighbors. This reduces noise when similarities are reliable but increases **variance** when few neighbors are available or similarities are weak.
- **Moderate \(k\) (e.g., 20–30):** Often gives the **best trade-off**—enough neighbors to stabilize estimates without averaging in dissimilar items.
- **Large \(k\) (e.g., 40–50):** More neighbors smooth predictions but can **dilute** true item–item relationships, pulling estimates toward the item mean and slightly **increasing** error if many weak neighbors enter the average.

In our run, inspect the line plot: RMSE/MAE typically decrease from \(k=10\) toward a minimum near \(k \approx 20\)–\(30\), then flatten or rise slightly as \(k\) grows. The default `K_NEIGHBORS=20` in `src/config.py` is consistent with this sensitivity analysis.

## 8. Conclusion

This project implemented and compared recommendation approaches on the **MovieLens 100K** dataset with a reproducible **80/20 hold-out** evaluation (`random_state=42`).

**Main findings:**

1. **Item-Based Collaborative Filtering** delivered the strongest results (RMSE **0.9381**, MAE **0.7342**), outperforming User-Based CF and a global-mean baseline.
2. **User-Based CF** remained competitive and demonstrated that leveraging neighborhood structure clearly beats ignoring personalization.
3. **TruncatedSVD** provided a fast matrix-factorization baseline but did not beat neighborhood CF without bias terms and tuned optimization.
4. **Data sparsity** (~93.7% missing entries) and **rating-level effects** (higher error on 1-star and 5-star ratings) explain remaining prediction error and guide future work (biased MF, hybrid models, or deep recommenders).

For a production system, we would add cross-validation, hyperparameter search, and ranking metrics (e.g., precision@k); for this course project, the experiments show a clear, interpretable progression from baseline → collaborative filtering → optional low-rank factorization.